# Reproduce *Digital Psychopharmacology* — full experiment playbook

One notebook that regenerates the paper's key experiments **and the collateral that supports them**,
then shows each artifact next to the figure/table/claim it backs. Driven by `scripts/reproduce.py`.

**Pick a compute tier** (cell below):

| Tier | Needs | Reproduces |
|---|---|---|
| **0** | CPU only | Validators + committed-data figures + the whole visual pipeline in dry-run form. No GPU/token. |
| **1** | 1 GPU | Real SDXL-Turbo dose-response (Table 2, ghost, safety, collapse, vitals) + text battery on **gpt2** (Table 1, Figs 4 & 6). |
| **2** | 1 GPU + HF token | Text experiments on the paper's gated **Llama-3.1-8B** at paper scale — the only tier that reproduces exact numbers. |

For a GPU: **Runtime → Change runtime type → GPU** (a free T4 is enough for tiers 1–2 on the 8B in 4-bit).
See `REPRODUCIBILITY.md` for the full artifact→claim map.

> **Runs three ways:** in Colab (open this file), in a **local Jupyter** (the clone/install/download cells auto-skip when you're already in the repo), or headless via `python scripts/reproduce.py --tier <n>` (see `REPRODUCIBILITY.md`).

## 1. Get the code + dependencies

In [ ]:
import os
# Colab: clone into /content. Local Jupyter already inside the repo: skip the clone/cd.
if not os.path.exists('scripts/reproduce.py'):
    if not os.path.isdir('neuromod-llm-poc'):
        !git clone -q https://github.com/cneckar/neuromod-llm-poc.git
    os.chdir('neuromod-llm-poc')
!git pull -q origin main 2>/dev/null; true
# Core package + visual/metric backends. (Tier 0 only strictly needs the base install.)
!pip install -q -e . 2>/dev/null; true
!pip install -q 'diffusers>=0.27' transformers accelerate open_clip_torch lpips scikit-image imageio seaborn statsmodels

## 2. Choose your tier
Set `TIER` below. Tier 2 also needs an HF token with Llama-3.1 access — paste it in the next cell.

In [ ]:
TIER = 1          # 0 = CPU/committed-data, 1 = GPU + gpt2, 2 = GPU + gated Llama-3.1-8B
SEEDS = 16        # visual seeds per dose (16 pilot, 100 for the full study)
INTENSITIES = '0.0,0.25,0.5,0.75,1.0'
print(f'Tier {TIER}, {SEEDS} seeds, doses {INTENSITIES}')

### (Tier 2 only) Unlock the gated Llama-3.1-8B
Accept the license at <https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct>, create a read token,
and paste it below. Leave blank for tiers 0–1 (which use gpt2).

In [ ]:
import os
from getpass import getpass
if TIER == 2:
    tok = getpass('HF token (blank to skip): ').strip()
    if tok:
        os.environ['HUGGINGFACE_TOKEN'] = tok
        os.environ['HF_TOKEN'] = tok
        try:
            from huggingface_hub import login; login(tok)
        except Exception as e:
            print('login skipped:', e)
    MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
else:
    MODEL = 'gpt2'
print('Text model:', MODEL)

## 3. Run the reproduction
Runs every stage for the chosen tier, writing all artifacts under `outputs/reproduction/` and a
`REPRODUCTION_REPORT.md` that maps each one to the paper claim it supports. Resumable + continues on
failure. First run downloads models (a few GB on tiers 1–2).

In [ ]:
!python scripts/reproduce.py --tier {TIER} --model {MODEL} --seeds {SEEDS} \
    --intensities {INTENSITIES} --outdir outputs/reproduction

## 4. The reproduction report
Every key experiment, its status, and the artifacts it produced.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('outputs/reproduction/REPRODUCTION_REPORT.md').read()))

## 5. Headline collateral — the visual dose-response findings
### 5a. Decision matrix — which thread is the talk headline?

In [ ]:
import pandas as pd, os
p='outputs/reproduction/pilot/decision_matrix.csv'
display(pd.read_csv(p)) if os.path.exists(p) else print('run tier >=0 first')

### 5b. The winning thread's hero montage (ghost montage / collapse grid)

In [ ]:
import glob
from IPython.display import Image, display
heroes = sorted(glob.glob('outputs/reproduction/pilot/hero_*.png'))
for h in heroes:
    print(h); display(Image(h))
if not heroes: print('no hero montage (headline thread has no montage builder, e.g. vitals)')

### 5c. Dose-response curves (CI ribbons, EC50, monotonicity)
Addresses the paper's own §6.2 *"Absence of Dose-Response and Monotonicity Curves"* limitation.

In [ ]:
import glob, pandas as pd, os
from IPython.display import Image, display
t='outputs/reproduction/dose_stats/trend_summary.csv'
if os.path.exists(t):
    df=pd.read_csv(t)
    cols=[c for c in ['pack','metric','trend','spearman_rho','spearman_q','ec50','breakpoint_dose'] if c in df.columns]
    display(df.sort_values('spearman_rho', key=abs, ascending=False)[cols].head(12))
for g in sorted(glob.glob('outputs/reproduction/dose_stats/plots/*latent_energy.png'))[:4]:
    display(Image(g))

### 5d. Cocaine Crunch — stimulant mode-collapse (Table 2 as curves)

In [ ]:
import pandas as pd, os
from IPython.display import Image, display
p='outputs/reproduction/mode_collapse/stimulant_phenotypes.csv'
display(pd.read_csv(p)) if os.path.exists(p) else print('n/a')
d='outputs/reproduction/mode_collapse/diversity_collapse.png'
display(Image(d)) if os.path.exists(d) else None

### 5e. Latent Specter (ghost vs placebo) & Architectural Jailbreak (safety) — tier 1+
The DMT-ghost anecdote as a statistical result, and §7.4 Spectral Safety Auditing as trigger-rate curves.

In [ ]:
import os
from IPython.display import Image, display
for f in ['outputs/reproduction/latent_specter/ghost_prevalence.png',
          'outputs/reproduction/safety_boundary/trigger_rates.png']:
    display(Image(f)) if os.path.exists(f) else print('(tier 1+)', f)

### 5f. Vitals monitor — the dose slider

In [ ]:
import os
from IPython.display import Image, display
g='outputs/reproduction/vitals/vitals_monitor.gif'
display(Image(g)) if os.path.exists(g) else print('n/a')

## 6. Text-model collateral
### 6a. Detection sensitivity (Fig 4), behavioral radar (Fig 6), emotion signatures (Fig 10)

In [ ]:
import os
from IPython.display import Image, display
for f in ['outputs/reproduction/figures/figure_2_detection_sensitivity.png',
          'outputs/reproduction/figures/figure_3_radar_plots.png',
          'outputs/reproduction/figures/figure_5_emotion_signatures.png']:
    print(f); display(Image(f)) if os.path.exists(f) else print('  (tier 1+ for 2&3; emotion is tier 0)')

### 6b. Primary-endpoint statistics (Table 1) — tier 1+

In [ ]:
import json, os
p='outputs/reproduction/analysis/statistical_analysis_results.json'
if os.path.exists(p):
    r=json.load(open(p)); print(json.dumps(r, indent=2)[:2000])
else:
    print('run tier >=1 (endpoints + analyze_endpoints) for Table 1 statistics')

## 7. Download everything

In [ ]:
!zip -qr reproduction.zip outputs/reproduction
try:
    from google.colab import files  # Colab: trigger a browser download
    files.download('reproduction.zip')
except Exception:
    import os
    print('Saved reproduction.zip ->', os.path.abspath('reproduction.zip'))
    print('(local run: artifacts are also under outputs/reproduction/)')